In [ ]:
from pathlib import Path

from weavehr import WeavEHRProject, ExtractionStep, ConceptStep, ShardingStep
# from weavehr import PersistenceStep
# from weavehr.logging import configure_logging

import weavehr_project  # noqa: F401

# configure_logging(level="DEBUG")

config_path = Path.cwd().parent / "configs"

In [ ]:
from weavehr.steps.concept.registry import concept_config_registry
from weavehr.steps.concept.config.concept import ConceptConfig
from weavehr.config.registry import load_configs
from weavehr.steps.extraction.registry import dataset_config_registry

# load custom dataset and concept configs
mapping_paths = []
for dataset_path in (config_path / "datasets").iterdir():
	for version_path in dataset_path.iterdir():
		if version_path.is_dir():
			dataset_config_registry.load(version_path / "tables")
			mapping_paths.append(version_path / "mappings")

concepts = load_configs(
	config_path / "concepts",
	config_type=ConceptConfig,
	dataset_paths=mapping_paths,
)
for concept in concepts:
	concept_config_registry.register(concept)

In [ ]:
pipe_config_path: Path = config_path / "pipeline"
project_path = Path.cwd().parent / "data" / "project"

with WeavEHRProject(project_path) as project:
    # # Optional: persist the source columns selected by extraction.yml as Parquet files.
    # # These files are independent artifacts and are not used by the ExtractionStep.
    # persistence_step = PersistenceStep.load(project, config_path / "extraction.yml")
    # persistence_step.run()

    extraction_step = ExtractionStep.load(project, pipe_config_path / "extraction.yml")
    extraction_step.run()

    concept_step = ConceptStep.load(project, pipe_config_path / "concept.yml")
    concept_step.run()

    sharding_step = ShardingStep.load(project, pipe_config_path / "sharding.yml")
    sharding_step.run()